# Estratégia de Reversão à Média — Parte 3: Exportação ONNX e Integração com MetaTrader 5

Este notebook cobre:
- Carregamento do modelo treinado
- Exportação para o formato ONNX
- Geração do arquivo de inclusão MQL5
- Instruções de configuração no MetaTrader 5

## 1. Configuração do Ambiente

In [ ]:
import sys
import os
import pickle

sys.path.insert(0, os.path.abspath('..'))

from export_lib import export_model_to_ONNX

print('Ambiente configurado.')

## 2. Hiperparâmetros

> Mantenha os mesmos valores dos notebooks anteriores.

In [ ]:
from datetime import datetime

hyper_params = {
    'symbol':       'EURGBP_H1',
    'model_number': 0,
    'periods':      [i for i in range(5, 300, 30)],
    'periods_meta': [10],
    'markup':       0.00010,
    'stop_loss':    0.02000,
    'take_profit':  0.00200,
    'backward':     datetime(2000, 1, 1),
    'forward':      datetime(2021, 1, 1),
    'n_clusters':   10,
    'rolling':      200,
}

## 3. Configuração do Caminho de Exportação

Defina abaixo o caminho para a pasta `Include/Mean Reversion/` dentro da sua instalação do MetaTrader 5.

**Como encontrar a pasta do MT5:**
1. Abra o MetaTrader 5
2. Menu `Arquivo` → `Abrir Pasta de Dados`
3. Navegue até `MQL5/Include/` e crie a pasta `Mean Reversion/` se não existir
4. Cole o caminho completo abaixo

Por padrão, o notebook exporta para a pasta `MQL5/Include/Mean Reversion/` do próprio repositório.

In [ ]:
# Caminho padrão: pasta MQL5 do repositório (relativo ao notebook)
export_path = os.path.abspath(
    os.path.join('..', '..', 'MQL5', 'Include', 'Mean Reversion') + os.sep
)

# Para exportar direto para o MetaTrader 5, substitua pelo caminho real:
# export_path = r'C:\Users\SEU_USUARIO\AppData\Roaming\MetaQuotes\Terminal\SEU_ID\MQL5\Include\Mean Reversion\'

# Garante que a pasta existe
os.makedirs(export_path, exist_ok=True)

print(f'Caminho de exportação: {export_path}')
print(f'A pasta existe: {os.path.isdir(export_path)}')

## 4. Carregamento do Modelo Treinado

Carrega o modelo salvo pelo notebook anterior. Caso você esteja executando os três notebooks em sequência na mesma sessão, pode usar diretamente o objeto `best_model` em memória — basta pular esta célula.

In [ ]:
model_path = os.path.join('..', 'best_model.pkl')

if os.path.exists(model_path):
    with open(model_path, 'rb') as f:
        best_model = pickle.load(f)
    print(f'Modelo carregado de: {os.path.abspath(model_path)}')
    print(f'R² do modelo: {best_model[0]:.4f}')
else:
    print('Arquivo best_model.pkl não encontrado.')
    print('Execute o notebook 02 primeiro, ou certifique-se de que best_model está em memória.')

## 5. Exportação para ONNX

A função `export_model_to_ONNX` gera três arquivos:
- `catmodel EURGBP_H1 0.onnx` — modelo principal (prediz direção: compra/venda)
- `catmodel_m EURGBP_H1 0.onnx` — meta-modelo (confirma o regime de mercado)
- `EURGBP_H1 ONNX include 0.mqh` — arquivo de inclusão MQL5 com definições de features

In [ ]:
export_model_to_ONNX(
    model        = best_model,
    symbol       = hyper_params['symbol'],
    periods      = hyper_params['periods'],
    periods_meta = hyper_params['periods_meta'],
    model_number = hyper_params['model_number'],
    export_path  = export_path
)

print('\nArquivos gerados:')
for f in os.listdir(export_path):
    size = os.path.getsize(os.path.join(export_path, f))
    print(f'  {f}  ({size/1024:.1f} KB)')

## 6. Verificação do Arquivo de Inclusão MQL5

Exibe o conteúdo do arquivo `.mqh` gerado para confirmar que os períodos e funções estão corretos.

In [ ]:
mqh_file = os.path.join(
    export_path,
    f"{hyper_params['symbol']} ONNX include {hyper_params['model_number']}.mqh"
)

with open(mqh_file, 'r') as f:
    content = f.read()

print(content)

## 7. Instruções para o MetaTrader 5

Após a exportação, siga os passos abaixo para configurar o Expert Advisor no MetaTrader 5.

### 7.1 Copiar os arquivos

Se você exportou para a pasta do repositório (não diretamente para o MT5), copie os arquivos:

```
Origem:  MQL5/Include/Mean Reversion/
Destino: [PASTA_DADOS_MT5]/MQL5/Include/Mean Reversion/
```

Copie também o Expert Advisor:

```
Origem:  MQL5/Mean Reversion.mq5
Destino: [PASTA_DADOS_MT5]/MQL5/Experts/
```

### 7.2 Compilar o EA

1. Abra o **MetaEditor** no MT5 (`F4`)
2. Navegue até `Experts/Mean Reversion.mq5`
3. Compile com `F7`
4. Verifique que não há erros

### 7.3 Executar Backtesting

1. Abra o **Strategy Tester** (`Ctrl+R`)
2. Selecione: EA = `Mean Reversion`, Símbolo = `EURGBP`, Timeframe = `H1`
3. Configure o período desejado e clique em **Iniciar**

### 7.4 Executar ao vivo (demo ou real)

1. Arraste o EA `Mean Reversion` para o gráfico `EURGBP H1`
2. Configure stop-loss e take-profit nos parâmetros de entrada
3. Habilite o trading automático (botão verde na barra de ferramentas)

## 8. Limpeza de Arquivos Temporários (Opcional)

In [ ]:
# Remove o arquivo pickle temporário após a exportação bem-sucedida
# Descomente se quiser apagar automaticamente

# if os.path.exists(model_path):
#     os.remove(model_path)
#     print('Arquivo temporário best_model.pkl removido.')

print('Exportação concluída com sucesso!')